# Consumer Discretionary: FinBERT scoring + Model C

Runtime > Change runtime type > **GPU** (T4 takes about 2 h at the IT run's ~190 sentences/s; L4/A100 is faster).

1. Put `cd_finbert_input.zip` in the root of My Drive. It holds licensed WRDS transcripts: keep it private and delete it afterwards.
2. Run all cells. Finished years are checkpointed to Drive, so after a disconnect just Run all again.
3. Download `MyDrive/cd_finbert/cd_finbert_output.zip` and unzip it into `cd_results/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install transformers xgboost pyarrow scikit-learn
!mkdir -p /content/cd /content/drive/MyDrive/cd_finbert
!unzip -qo /content/drive/MyDrive/cd_finbert_input.zip -d /content/cd
!ls /content/cd /content/cd/earnings_sp500/text

In [ ]:
# FinBERT features (same script and settings as the IT run)
!cd /content/cd && python it_finbert_features.py --sp500-folder earnings_sp500 --it-folder earnings_cd --batch-size 128 --checkpoint-dir /content/drive/MyDrive/cd_finbert/ckpt 2>&1 | tee -a earnings_cd/cd_finbert_run.log
import os
assert os.path.exists("/content/cd/earnings_cd/finbert_features.parquet"), "FinBERT step failed: read the log above"

In [ ]:
# Check before modelling
import pandas as pd
f = pd.read_parquet("/content/cd/earnings_cd/finbert_features.parquet")
print(f.shape, f.transcriptid.is_unique)
assert len(f) == 3710 and f.transcriptid.is_unique
print(f.describe().T)

In [ ]:
# Model C (includes XGBoost, as in the IT run)
!cd /content/cd && python it_model_c_finbert.py --folder earnings_cd --prefix cd --sector-label CD 2>&1 | tee earnings_cd/cd_modelc_run.log
import glob
assert glob.glob("/content/cd/earnings_cd/cd_model_c_*_report.txt"), "Model C failed: read the log above"

In [ ]:
import pandas as pd
f = pd.read_parquet("/content/cd/earnings_cd/finbert_features.parquet")
f.to_csv("/content/cd/earnings_cd/finbert_features.csv", index=False)
!cd /content/cd/earnings_cd && zip -q /content/drive/MyDrive/cd_finbert/cd_finbert_output.zip finbert_features.* cd_model_c_* cd_finbert_run.log cd_modelc_run.log
!unzip -l /content/drive/MyDrive/cd_finbert/cd_finbert_output.zip